# e-SNLI — Transcoder Analysis with Attention Freezing + Per-Token Feature Ablation

## Imports

In [1]:
import sys, os, torch, pandas as pd
from functools import partial
sys.path.insert(0, os.path.dirname(os.getcwd()))

from huggingface_hub import hf_hub_download, login
from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display

from src.configs import DatasetConfig, InferenceConfig, PromptStyle
from src.dataset.esnli import ESNLI_Dataset
from src.SAE import JumpReLUSAE
from src.neuronpedia_client import NeuronpediaClient

## Configuration

In [2]:
LAYER     = 31
WIDTH     = "262k"
L0        = "small"
REPO_ID   = "google/gemma-scope-2-27b-it"
TC_PATH   = f"transcoder/layer_{LAYER}_width_{WIDTH}_l0_{L0}_affine/params.safetensors"
THRESHOLD = 1   # min tokens a feature must fire on to be included in analysis
TOP_N     = 20  # top features to ablate (by max activation)

print(f"Model:        google/gemma-3-27b-it")
print(f"TC layer:     {LAYER}")
print(f"TC width:     {WIDTH}")
print(f"TC l0:        {L0}")
print(f"TC path:      {TC_PATH}")
print(f"Threshold:    {THRESHOLD} token(s)")
print(f"Top-N:        {TOP_N}")

Model:        google/gemma-3-27b-it
TC layer:     31
TC width:     262k
TC l0:        small
TC path:      transcoder/layer_31_width_262k_l0_small_affine/params.safetensors
Threshold:    1 token(s)
Top-N:        20


## HF Token

In [3]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

## Load Dataset + Build Prompts

In [4]:
dataset_config = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

inference_config = InferenceConfig(
    batch_size=2,
    max_new_tokens=512,
    downsample_rate=100,
)

esnli_dataset = ESNLI_Dataset(dataset_config)
prompted_data = esnli_dataset.build_prompts()
if inference_config.downsample_rate > 1:
    n = max(1, len(prompted_data) // inference_config.downsample_rate)
    prompted_data = prompted_data.shuffle(seed=42).select(range(n))
esnli_df = prompted_data.to_pandas()
print(f"Dataset size: {len(esnli_df)}")
esnli_df.head()

Data successfully loaded.
Dataset size: 98


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label,prompt
0,Two people sit facing away in a downtown scene...,The two people run as quickly as they can for ...,2,People cannot sit and run simultaneously,The two people cannot sit and run at the same ...,People cannot run and sit simultaneously. Poo...,contradiction,<start_of_turn>user Task: Determine the logica...
1,A white dog with brown ears runs down a gravel...,A dog runs down a path with a green ball.,1,"Not all balls are green, the dog has a ball, b...",The ball is not necessarily green.,Not all balls are green.,neutral,<start_of_turn>user Task: Determine the logica...
2,"Six men, all wearing identifying number plaque...",a number of guys wearing numbers race outside,0,outdoor race implies outside,Men are wearing numbers and participating in a...,"Six men is a number of guys, and race outside ...",entailment,<start_of_turn>user Task: Determine the logica...
3,Five children of Indian origin are smiling and...,Children are on a slide.,0,They are on a slide because they are posing on...,Children are on a slide is a simplification of...,Both sentences are about children on a slide.,entailment,<start_of_turn>user Task: Determine the logica...
4,Kids are on a amusement ride.,Kids ride their favorite amusement ride.,1,It isn't necessarily their favorite ride.,Being on a amusement ride doesn't imply ride o...,Not every amusement ride will be the kids favo...,neutral,<start_of_turn>user Task: Determine the logica...


## Load Model + Transcoder

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-27b-it", device_map=device, dtype=torch.bfloat16
)
model.eval()

path_to_params = hf_hub_download(repo_id=REPO_ID, filename=TC_PATH)
params = load_file(path_to_params)
d_model, d_sae = params["w_enc"].shape
print(f"d_model={d_model}, d_sae={d_sae}")

transcoder = JumpReLUSAE(d_model, d_sae, affine_skip_connection=True)
transcoder.load_state_dict(params)
transcoder = transcoder.to(device=device, dtype=torch.float32).eval()
print("Transcoder loaded.")

num_layers = len(model.model.language_model.layers)
print(f"num_layers={num_layers}")

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

d_model=5376, d_sae=262144
Transcoder loaded.
num_layers=62


## Random Sample + Prompt

In [6]:
sample        = esnli_df.sample(1, random_state=None)
sample_prompt = sample["prompt"].item()
sample_label  = sample["gold_label"].item()
print(f"Sample Index: {sample.index.tolist()}")
print(f"Label: {sample_label}")
print()
print(sample_prompt)

Sample Index: [94]
Label: entailment

<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A girl standing by a decorated structural beam poses for a picture.
Hypothesis: A girl poses for a picture.

<end_of_turn>model 


## Generate Baseline Response (Autoregressive)

Standard `model.generate(do_sample=False)` with no hooks. Stores `full_ids`, `prompt_len`,
and `gen_len` for use in subsequent cells.

In [7]:
inputs = tokenizer(sample_prompt, return_tensors="pt", add_special_tokens=True).to(model.device)
prompt_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=inference_config.max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

full_ids = output_ids          # (1, prompt_len + gen_len)
gen_len  = output_ids.shape[1] - prompt_len

print(f"prompt_len={prompt_len}, gen_len={gen_len}, total={full_ids.shape[1]}")
print()
print(tokenizer.decode(full_ids[0], skip_special_tokens=False))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


prompt_len=103, gen_len=108, total=211

<bos><start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A girl standing by a decorated structural beam poses for a picture.
Hypothesis: A girl poses for a picture.

<end_of_turn>model 
<reasoning>
The premise states a specific scenario: a girl posing for a picture *by a decorated structural beam*. The hypothesis states a more general scenario: a girl posing for a picture. If the premise is true, the hypothesis *must* also be true. The specific details in the premise (the beam, the decoration) don't change the fact that a girl is posing for a picture. Therefore, the premise entails the hypothesis.
</reasoning>
<label>entailment</label><end_of_turn>


## Full-Sequence Forward Pass: Capture MLP Inputs + Future Attention Outputs

Two sets of hooks:
- **Hook A** (`pre_feedforward_layernorm` at `layers[LAYER]`): captures the transcoder input, shape `(n_tokens, d_model)`.
- **Hooks B** (`self_attn` at `layers[L]` for `L > LAYER`): captures attention outputs for freezing during ablation.

In Gemma 3's dual-layernorm architecture:
```
x_normed   = pre_feedforward_layernorm(residual)   # ← Hook A reads here
mlp_out    = mlp(x_normed)
normed_out = post_feedforward_layernorm(mlp_out)
residual   = residual + normed_out
```

In [8]:
cache      = {}
attn_cache = {}  # attn_cache[L] = (1, n_tokens, d_model)
handles    = []

def hook_capture_mlp_in(module, inp, out):
    cache["mlp_in"] = out.detach().squeeze(0)  # (n_tokens, d_model)
    return out

def make_capture_attn(layer_idx):
    def hook(module, inp, out):
        # self_attn returns (attn_output, attn_weights_or_None, past_kv_or_None)
        attn_cache[layer_idx] = out[0].detach().clone()  # (1, n_tokens, d_model)
        return out
    return hook

try:
    layer = model.model.language_model.layers[LAYER]
    handles.append(
        layer.pre_feedforward_layernorm.register_forward_hook(hook_capture_mlp_in)
    )
    for L in range(LAYER + 1, num_layers):
        handles.append(
            model.model.language_model.layers[L].self_attn.register_forward_hook(
                make_capture_attn(L)
            )
        )

    with torch.no_grad():
        model(input_ids=full_ids)
finally:
    for h in handles:
        h.remove()

mlp_in_acts = cache["mlp_in"]  # (n_tokens, d_model)
n_tokens = mlp_in_acts.shape[0]
print(f"mlp_in_acts.shape:           {mlp_in_acts.shape}")
print(f"attn_cache layers captured:  {sorted(attn_cache.keys())}")
print(f"attn_cache[{LAYER+1}].shape: {attn_cache[LAYER+1].shape}")

mlp_in_acts.shape:           torch.Size([211, 5376])
attn_cache layers captured:  [32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
attn_cache[32].shape: torch.Size([1, 211, 5376])


## Transcoder Encoding

In [9]:
with torch.no_grad():
    tc_acts_full = transcoder.encode(mlp_in_acts.float())   # (n_tokens, d_sae)

tc_acts_gen = tc_acts_full[prompt_len:]   # generated tokens only
tokens      = tokenizer.convert_ids_to_tokens(full_ids[0])

print(f"Transcoder activations (full):     {tc_acts_full.shape}")
print(f"Transcoder activations (gen-only): {tc_acts_gen.shape}")
print(f"L0 (full):  {(tc_acts_full > 0).float().sum(dim=-1).mean():.1f}")
print(f"L0 (gen):   {(tc_acts_gen > 0).float().sum(dim=-1).mean():.1f}")

Transcoder activations (full):     torch.Size([211, 262144])
Transcoder activations (gen-only): torch.Size([108, 262144])
L0 (full):  22.0
L0 (gen):   20.7


## Feature Analysis Table

Show all features active on at least `THRESHOLD` tokens across the **full sequence**,
sorted by descending max activation. Fetch Neuronpedia labels.

In [10]:
# Per-feature stats over the full sequence
token_count = (tc_acts_full > 0).sum(dim=0)                            # (d_sae,)
max_act     = tc_acts_full.max(dim=0).values                           # (d_sae,)
avg_act     = tc_acts_full.sum(dim=0) / token_count.clamp(min=1)      # mean over active tokens

# Filter: features active on >= THRESHOLD tokens
valid_mask = token_count >= THRESHOLD
valid_idxs = valid_mask.nonzero(as_tuple=False).squeeze(-1)
print(f"Features active on >= {THRESHOLD} token(s): {valid_mask.sum().item()}")

# Sort valid features by max_act descending, take top TOP_N
valid_max_act = max_act[valid_idxs]
sorted_order  = valid_max_act.argsort(descending=True)[:TOP_N]
sorted_idxs   = valid_idxs[sorted_order].tolist()

# Active token strings per feature
active_tokens_per_feat = {
    fi: [tokens[t] for t in (tc_acts_full[:, fi] > 0).nonzero(as_tuple=True)[0].tolist()]
    for fi in sorted_idxs
}

# Fetch Neuronpedia labels
np_model_id = "gemma-3-27b-it"
np_sae_id   = f"{LAYER}-gemmascope-2-transcoder-{WIDTH}"
client      = NeuronpediaClient(model_id=np_model_id, sae_id=np_sae_id)
np_features = client.get_features(sorted_idxs)

rows = [
    {
        "Feature IDX":          fi,
        "Max Act":              round(max_act[fi].item(), 4),
        "Avg Act":              round(avg_act[fi].item(), 4),
        "Tokens Active":        token_count[fi].item(),
        "Active Token Strings": active_tokens_per_feat[fi],
        "Description":          (np_features[fi].description or "N/A") if fi in np_features else "N/A",
    }
    for fi in sorted_idxs
]
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)
display(pd.DataFrame(rows))

Features active on >= 1 token(s): 2253


,Feature IDX,Max Act,Avg Act,Tokens Active,Active Token Strings,Description
0,5628,5094.3354,5094.3354,1,[<bos>],bullet points or lists
1,6321,5032.2539,5032.2539,1,[<bos>],"code, multilingual, principles"
2,2362,5006.7256,5006.7256,1,[<bos>],lists with asterisks and letters
3,4302,4934.9888,4934.9888,1,[<bos>],list items or enumerations
4,1581,4903.7505,4903.7505,1,[<bos>],list item markers and non-English characters
5,7544,4796.6333,4796.6333,1,[<bos>],"time, seconds, minutes, hours"
6,7204,4730.5107,4730.5107,1,[<bos>],code documentation examples
7,5986,4653.1191,4653.1191,1,[<bos>],multilingual words
8,6957,4626.4180,4626.4180,1,[<bos>],N/A
9,1757,4597.6113,4597.6113,1,[<bos>],time or duration


## Ablation Loop with Frozen Attention (Single-Pass)

### Methodology

For each transcoder feature we want to ablate, we run a **single forward pass** on the
full token sequence (prompt + generated tokens) with two sets of hooks active:

1. **Per-token feature ablation** (dual-hook at `layers[LAYER]`)
   - Hook A reads `pre_feedforward_layernorm` output → encodes with the transcoder.
   - The target feature is zeroed at **every token position**.
   - The delta `transcoder.decode(acts_ablated) - transcoder.decode(acts)` is injected
     via a hook on `post_feedforward_layernorm` output.

2. **Frozen attention** at all layers `L > LAYER`
   - We replace each `self_attn` output with the cached value from the baseline
     forward pass (captured in Cell 9).
   - This isolates the causal effect of the transcoder feature from downstream
     attention re-routing.

### Why single-pass (not `generate()`)?

- Avoids `HybridCache` vs `DynamicCache` incompatibility in Gemma 3 + transformers ≥ 5.x.
- Attention patterns are fixed for the full sequence; autoregressive generation with
  frozen attention would require step-by-step positional bookkeeping.
- Single-pass causal ablation (activation patching) is the standard approach in the
  mechanistic interpretability literature.

The **steered output** is obtained by greedy-decoding the ablated logits:
`ablated_tokens[t]` predicts the token at position `t+1`, so we shift by one when
decoding for display.

In [11]:
def run_ablation(feature_idx: int) -> torch.Tensor:
    """
    Returns logits (n_tokens, vocab) from a forward pass where:
      - feature `feature_idx` is zeroed in transcoder latent space at LAYER
      - self_attn outputs are frozen at layers LAYER+1..num_layers-1
    """
    read_buf = {}
    handles  = []

    def hook_read_mlp(module, inp, out):
        read_buf["mlp_in"] = out   # (1, seq, d_model) in bfloat16
        return out

    def hook_inject_mlp(module, inp, out):
        mlp_in       = read_buf["mlp_in"].float()             # (1, seq, d_model)
        acts         = transcoder.encode(mlp_in)              # (1, seq, d_sae)
        acts_ablated = acts.clone()
        acts_ablated[..., feature_idx] = 0
        delta_mlp = (
            transcoder.decode(acts_ablated) - transcoder.decode(acts)
        )  # (1, seq, d_model)
        return out + delta_mlp.to(dtype=out.dtype)

    def make_freeze_attn(cached):
        def hook(module, inp, out):
            # out = (attn_output, attn_weights_or_None, past_kv_or_None)
            return (cached.to(dtype=out[0].dtype), *out[1:])
        return hook

    # Register hooks
    layer = model.model.language_model.layers[LAYER]
    handles.append(
        layer.pre_feedforward_layernorm.register_forward_hook(hook_read_mlp)
    )
    handles.append(
        layer.post_feedforward_layernorm.register_forward_hook(hook_inject_mlp)
    )
    for L in range(LAYER + 1, num_layers):
        handles.append(
            model.model.language_model.layers[L].self_attn.register_forward_hook(
                make_freeze_attn(attn_cache[L])
            )
        )

    try:
        with torch.no_grad():
            out = model(input_ids=full_ids)
        return out.logits.squeeze(0)   # (n_tokens, vocab)
    finally:
        for h in handles:
            h.remove()


print("run_ablation() defined.")
print(f"Will ablate top {TOP_N} features from sorted_idxs.")

run_ablation() defined.
Will ablate top 20 features from sorted_idxs.


## Baseline Logits

Single-pass baseline (no hooks) for comparison against ablated runs.

In [12]:
with torch.no_grad():
    baseline_logits = model(input_ids=full_ids).logits.squeeze(0)  # (n_tokens, vocab)

print(f"baseline_logits.shape: {baseline_logits.shape}")

# Sanity check: shapes
assert mlp_in_acts.shape == (n_tokens, d_model),      f"mlp_in_acts shape mismatch: {mlp_in_acts.shape}"
assert tc_acts_full.shape == (n_tokens, d_sae),       f"tc_acts_full shape mismatch: {tc_acts_full.shape}"
assert attn_cache[LAYER + 1].shape == (1, n_tokens, d_model), (
    f"attn_cache shape mismatch: {attn_cache[LAYER+1].shape}"
)
print("All shape sanity checks passed.")

baseline_logits.shape: torch.Size([211, 262208])
All shape sanity checks passed.


## Results: Per-Feature Ablation

For each of the top `TOP_N` features (by max activation):
1. Run `run_ablation(fi)` → ablated logits.
2. Greedy-decode both baseline and ablated logits.
3. Print the full-sequence predicted text for each and highlight token-level differences.

Token shift: `logits[t]` predicts position `t+1`, so `ablated_tokens[:-1]`
corresponds to predicted tokens for positions `1..n_tokens`.

In [13]:
baseline_pred_ids = baseline_logits.argmax(dim=-1)  # (n_tokens,)

# Build a label lookup from the already-fetched Neuronpedia results
feat_labels = {
    fi: (np_features[fi].description or "N/A") if fi in np_features else "N/A"
    for fi in sorted_idxs
}

for fi in sorted_idxs[:TOP_N]:
    ablated_logits  = run_ablation(fi)                  # (n_tokens, vocab)
    ablated_pred_ids = ablated_logits.argmax(dim=-1)    # (n_tokens,)

    # Shift: logits[t] -> token at position t+1
    # baseline ground truth: full_ids[0, 1:]  (what the model saw)
    # baseline prediction:   baseline_pred_ids[:-1]
    # ablated prediction:    ablated_pred_ids[:-1]
    baseline_text = tokenizer.decode(full_ids[0, 1:],          skip_special_tokens=False)
    steered_text  = tokenizer.decode(ablated_pred_ids[:-1],    skip_special_tokens=False)

    # Count differing token positions
    n_positions  = ablated_pred_ids[:-1].shape[0]
    n_diff       = (ablated_pred_ids[:-1] != baseline_pred_ids[:-1]).sum().item()

    label = feat_labels.get(fi, "N/A")
    header = f" Feature {fi} | max_act={max_act[fi].item():.1f} | \"{label}\" "
    print(f"{header:=^100}")
    print(f"[BASELINE] {baseline_text[:500]}")
    print(f"[STEERED ] {steered_text[:500]}")
    print(f"Differing positions: {n_diff} / {n_positions} tokens")
    print()

===================== Feature 5628 | max_act=5094.3 | "bullet points or lists" =====================
[BASELINE] <start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A girl standing by a decorated structural beam poses for a picture.
Hypothesis: A girl poses for a picture.

<end_of_turn>model 
<reasoning>
The premise states a specific scenari
[STEERED ] [,:: Write the the next between the setise and a Conclusion.

##:
ment, contradiction, consistency



Prem:
*. ** are answer a answer for thereasoning> and.
2. You MUST select your final answer ( <label> tags>.

3. You Prem MUST be before the label.

Premise: All person is near a tree Christmas element is for a photograph.
Hypothesis: A person is for a picture.
<


<reasoni